In [3]:
!pip install ipywidgets

   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   ----------------- ---------------------- 61.4/139.8 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 139.8/139.8 kB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/216.6 kB ? eta -:--:--
   --------------------------------------- 216.6/216.6 kB 13.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------- -------------------------------- 0.4/2.2 MB 13.5 MB/s eta 0:00:01
   ---------- ----------------------------- 0.6/2.2 MB 7.5 MB/s eta 0:00:01
   ------------- -------------------------- 0.7/2.2 MB 5.9 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.2 MB 6.0 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.2 MB 6.0 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.2 MB 3.4 MB/s eta 0:00:01
   ------------------------ --------------- 1.4/2.2 MB 5.1 MB/s eta 0:00:01
   --------------


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\Asus\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Ruthvik\Gov_Support_RAG_Chatbot\ministry_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
import pypdf

In [4]:
# function to load data files/ pdfs 
def pdf_file_loader(file):
    pdf_loader = DirectoryLoader(file, glob=["*.pdf"], loader_cls=PyPDFLoader) 
    
    pdf_docs = pdf_loader.load()

    return pdf_docs


In [20]:
data = pdf_file_loader(file= r"\Users\Ruthvik\Gov_Support_RAG_Chatbot\dataset")

In [21]:
data

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-07-01T11:20:33+00:00', 'source': '\\Users\\Ruthvik\\Gov_Support_RAG_Chatbot\\dataset\\Indian Constitution.pdf', 'total_pages': 402, 'page': 0, 'page_label': '1'}, page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ [1, 2024]THE CONSTITUTION OF INDIA[As on 1stMay, 2024] 2024GOVERNMENT OF INDIAMINISTRY OF LAW AND JUSTICELEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-07-01T11:20:33+00:00', 'source': '\\Users\\Ruthvik\\Gov_Support_RAG_Chatbot\\dataset\\Indian Constitution.pdf', 'total_pages': 402, 'page': 1, 'page_label': '2'}, page_content='PREFACEThis is the  sixth pocket size edition of the Constitution of India in the diglot form. In this edition, the text of the Constitution of India has been brought up-to-date by incorporating therein all the amendments up to the Constitution (One Hundred and S

In [ ]:
# Text chunking

def text_chunk(data):
    text_chunker = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    chunks = text_chunker.split_documents(data)

    return chunks

In [16]:
String_bits = text_chunk(data)

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings 

In [28]:
def HGF_embedder(model: str= "sentence-transformers/all-MiniLM-L6-v2"):
    embedding_model = HuggingFaceEmbeddings(model_name= model)

    return embedding_model

In [29]:
text_embedding_model = HGF_embedder()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
#from langchain.vectorstores.cassandra import Cassandra

In [33]:
ASTRA_DB_TOKEN = os.environ.get("ASTRA_DB_TOKEN")
ASTRA_DB_API = os.environ.get("ASTRA_DB_API")

In [ ]:
# # --- 4. Create Embeddings and Vector Store ---
# # GoogleGenerativeAIEmbeddings is the class for Gemini embeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # Recommended embedding model for Gemini

# # Create a FAISS vector store (in-memory for this example)
# vector_store = FAISS.from_documents(docs, embeddings)
# print("Vector store created and populated.")


None


In [ ]:
from langchain_astradb import AstraDBVectorStore

vstore = AstraDBVectorStore(
    collection_name="DocStore",
    embedding = text_embedding_model,
    token= "",
    api_endpoint= ""
)

In [ ]:
vstore.add_documents(String_bits)

### Searching on the vector db

In [ ]:
!pip install "astrapy>=2.0,<3.0"

In [ ]:
from astrapy import DataAPIClient

# Get an existing collection
client = DataAPIClient()
database = client.get_database(
    "API_ENDPOINT",
    token="APPLICATION_TOKEN",
)
collection = database.get_collection("DocStore")

# Find documents
search_query = "Explain the constitution of India"

# Perform the semantic search using the built-in vectorize
search_results = collection.find(
    sort={"$vectorize": search_query},
    limit=3,
    include_similarity=True
)

# Print the results
for doc in results["data"]["documents"]:
    print(doc.get('title', 'Untitled Document'))

ModuleNotFoundError: No module named 'astrapy.db'

### Prompting through LLM

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
     """ You are an assistant for question-answering tasks.
     "Use the following places of retrieved content to answer the question.
     if you don't know the answer, say that you are not aware of that.
     Use three sentence maximum and keep the answer concise."""
     "\n\n"
     "{context}"
)

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm=, prompt=)
rag_chain = create_retrieval_chain(search_results, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input" : "What is Acticle 73 about in constitution of India?"})
print(response["answer"])